In [1]:
import os
import glob
import gc
import random

from tqdm import tqdm

import numpy as np
import scipy.io
from scipy.sparse import vstack
import scipy.sparse as sp

import pandas as pd

import matplotlib.pyplot as plt

## Monkey Patching scVelos' version mismatch with matplotlib ##

# monkey patch removed attribute
import matplotlib.cbook as cbook

# monkey patch removed attribute
if not hasattr(cbook, "mplDeprecation"):
    import warnings
    cbook.mplDeprecation = DeprecationWarning 
    
## -- Monkey Patch Completed ##

import seaborn as sns

import scanpy as sc
import scvelo as scv

import pegasus as pg
from pegasusio import UnimodalData

%matplotlib inline

sc.settings.n_jobs = 25

random.seed(0)
np.random.seed(0)
sc.settings.seed = 0

In [2]:
## PATHS ##

MANUSCRIPTS = ['shin', 'sawada', 'purcell', 'walsh', 'rao', 'notaras', 'fernando', 'sebastian']

MANUSCRIPT_SRR_PATHS = {'shin': glob.glob('/mnt/sdb/scz_meta_analysis_processed/studies/shin_nowakowski/star_outputs/*/Velocyto/raw/*'),
                        'sawada': glob.glob('/mnt/sdb/scz_meta_analysis_processed/studies/sawada_kato/star_outputs/*/Velocyto/raw/*'),
                        'purcell': glob.glob('/mnt/sdb/scz_meta_analysis_processed/studies/purcell_mulle/star_outputs/*/Velocyto/raw/*'),
                        'walsh': glob.glob('/mnt/sdb/scz_meta_analysis_processed/studies/walsh_studer/star_outputs/*/Velocyto/raw/*'),
                        'rao': glob.glob('/mnt/sdb/scz_meta_analysis_processed/studies/walsh_studer/star_outputs/*/Velocyto/raw/*'),
                        'notaras': glob.glob('/mnt/sdb/scz_meta_analysis_processed/studies/notaras_colak/star_outputs/*/Velocyto/raw/*'),
                        'fernando': glob.glob('/mnt/lacie/scz_meta_analysis/fernando_brennand_nature_2025/star_outputs/*/Velocyto/raw/*'),
                        'sebastian': glob.glob('/mnt/lacie/scz_meta_analysis/sebastian_pak_nat_comm_2023/star_outputs/*/Velocyto/raw/*')}

MANUSCRIPT_SRR_SOLO_PATHS = {'shin': ["/".join(i.split('/')[0:8]) for i in MANUSCRIPT_SRR_PATHS['shin']],
                        'sawada': ["/".join(i.split('/')[0:8]) for i in MANUSCRIPT_SRR_PATHS['sawada']],
                        'purcell': ["/".join(i.split('/')[0:8]) for i in MANUSCRIPT_SRR_PATHS['purcell']],
                        'walsh': ["/".join(i.split('/')[0:8]) for i in MANUSCRIPT_SRR_PATHS['walsh']],
                        'rao': ["/".join(i.split('/')[0:8]) for i in MANUSCRIPT_SRR_PATHS['rao']],
                        'notaras': ["/".join(i.split('/')[0:8]) for i in MANUSCRIPT_SRR_PATHS['notaras']],
                        'fernando': ["/".join(i.split('/')[0:7]) for i in MANUSCRIPT_SRR_PATHS['fernando']],
                        'sebastian': ["/".join(i.split('/')[0:7]) for i in MANUSCRIPT_SRR_PATHS['sebastian']]}


In [3]:
## Iterate Over Each Manuscript ##

for manuscript in tqdm(MANUSCRIPTS):

# for manuscript in tqdm(MANUSCRIPTS[-1::]):

    print(f'Working on {manuscript}')
    
    # spliced and unspliced read in #
    spliced = vstack([scipy.io.mmread(i+'/Velocyto/raw/spliced.mtx').T.tocsr() for i in MANUSCRIPT_SRR_SOLO_PATHS[manuscript]])
    unspliced = vstack([scipy.io.mmread(i+'/Velocyto/raw/unspliced.mtx').T.tocsr() for i in MANUSCRIPT_SRR_SOLO_PATHS[manuscript]])
    
    # ensure all features the same #
    features = [pd.read_table(i+'/Velocyto/raw/features.tsv', header=None) for i in MANUSCRIPT_SRR_SOLO_PATHS[manuscript]]
    gene_sets = [set(f[1].values) for f in features]
    all_equal = all(g == gene_sets[0] for g in gene_sets)
    assert all_equal
    
    # import barcodes #
    barcodes = [pd.read_table(i+'/Velocyto/raw/barcodes.tsv', header=None).assign(library=i.split('/')[-1].split('_Solo.out')[0]) for i in MANUSCRIPT_SRR_SOLO_PATHS[manuscript]]
    barcodes = pd.concat(barcodes, axis=0)
    barcodes.rename(columns={0:'barcodes'}, inplace=True)
    barcodes['new_barcodes'] = barcodes['library'] + '_' + barcodes['barcodes']
    
    # make andata object #
    manuscript_velo = sc.AnnData(X=spliced)
    manuscript_velo.layers['spliced'] = spliced
    manuscript_velo.layers['unspliced'] = unspliced
    manuscript_velo.obs_names = barcodes['new_barcodes']
    manuscript_velo.var = features[0]
    manuscript_velo.var.index = features[0][1].values
    manuscript_velo.var.columns = ['ensembl_gene_ids', 'hgnc_gene_names', 'assay']

    # write h5ad #
    manuscript_velo.write_h5ad(f'/mnt/lacie/scz_meta_analysis/velocity_anndatas/velocity_{manuscript}.h5ad')

    del spliced
    del unspliced
    del barcodes
    del features
    del manuscript_velo
    gc.collect()

In [3]:
## Write Sebastian Libraries To Disk ##

manuscript = MANUSCRIPTS[-1]
paths = MANUSCRIPT_SRR_SOLO_PATHS[manuscript]

out_dir = f"/mnt/lacie/scz_meta_analysis/velocity_chunks/{manuscript}"
os.makedirs(out_dir, exist_ok=True)

for p in tqdm(paths):

    sample = p.split("/")[-1].split("_Solo.out")[0]

    spliced = scipy.io.mmread(p + "/Velocyto/raw/spliced.mtx").T.tocsr()
    unspliced = scipy.io.mmread(p + "/Velocyto/raw/unspliced.mtx").T.tocsr()

    # fast barcode read
    with open(p + "/Velocyto/raw/barcodes.tsv") as f:
        barcodes = np.array([f"{sample}_{line.strip()}" for line in f])

    # gene ref
    features = pd.read_csv(p + "/Velocyto/raw/features.tsv", header=None, sep="\t")
    genes = features[1].values

    ad = sc.AnnData(X=spliced)
    ad.layers["spliced"] = spliced
    ad.layers["unspliced"] = unspliced
    ad.obs_names = barcodes
    ad.var = pd.DataFrame(index=genes)

    ad.write_h5ad(f"{out_dir}/{sample}.h5ad")

    del ad, spliced, unspliced
    gc.collect()

100%|███████████████████████████████████████████████| 380/380 [1:17:59<00:00, 12.32s/it]


In [4]:
## Filter Sebastian Libraries Based on Barcode Inclusivity ##

# extract master barcode list ##
scz_adata = sc.read_h5ad('/mnt/sdb/scz_meta_analysis_processed/anndata_objs/integrated_adata_annotated_w_celltypist_scanvi.h5ad',
                         backed='r')

master_barcodes = scz_adata.obs_names
master_set = set(master_barcodes)

# create sebastian h5ad library paths #
in_dir = f"/mnt/lacie/scz_meta_analysis/velocity_chunks/sebastian/"
files = [os.path.join(in_dir, f) for f in os.listdir(in_dir) if f.endswith(".h5ad")]
print(f"{len(files)} files to process")

adata_out = None  # will grow incrementally

# loop over files
for f in tqdm(files):

    ad = sc.read_h5ad(f, backed='r')

    if ad.obs_names.isin(master_set).sum() > 0:
        ad = ad[ad.obs_names.isin(master_set)].to_memory()
    else:
        del ad
        gc.collect()
        continue

    # Remove duplicate genes in var_names
    if not ad.var.index.is_unique:
        print(f"Duplicate var_names found in {f}, cleaning...")
        ad = ad[:, ~ad.var.index.duplicated(keep='first')].copy() 
        gc.collect()

    ad.write_h5ad(f.split('.h5ad')[0] + '_filtered.h5ad')
    del ad
    gc.collect()

228 files to process


  1%|▍                                                  | 2/228 [00:11<22:26,  5.96s/it]

Duplicate var_names found in /mnt/lacie/scz_meta_analysis/velocity_chunks/sebastian/SRR23984482.h5ad, cleaning...


  2%|█                                                  | 5/228 [00:33<21:40,  5.83s/it]

Duplicate var_names found in /mnt/lacie/scz_meta_analysis/velocity_chunks/sebastian/SRR23984489_filtered_filtered.h5ad, cleaning...


...

 98%|████████████████████████████████████████████████▏| 224/228 [40:00<01:06, 16.66s/it]

Duplicate var_names found in /mnt/lacie/scz_meta_analysis/velocity_chunks/sebastian/SRR23984489.h5ad, cleaning...


 99%|████████████████████████████████████████████████▌| 226/228 [40:25<00:27, 13.57s/it]

Duplicate var_names found in /mnt/lacie/scz_meta_analysis/velocity_chunks/sebastian/SRR23984472.h5ad, cleaning...


100%|█████████████████████████████████████████████████| 228/228 [40:48<00:00, 10.74s/it]


In [5]:
## Try and Concatenate Sebastian Libraries ##

filtered_velocity_ads = [sc.read_h5ad(i) for i in glob.glob('/mnt/lacie/scz_meta_analysis/velocity_chunks/sebastian/*filtered*.h5ad')]

sebastian_filtered_velocity = sc.concat(filtered_velocity_ads, axis=0)

sebastian_filtered_velocity.write_h5ad('/mnt/lacie/scz_meta_analysis/velocity_anndatas/velocity_sebastian.h5ad')

In [2]:
glob.glob("/mnt/lacie/scz_meta_analysis/velocity_anndatas/*.h5ad")

['/mnt/lacie/scz_meta_analysis/velocity_anndatas/velocity_sawada.h5ad',
 '/mnt/lacie/scz_meta_analysis/velocity_anndatas/velocity_purcell.h5ad',
 '/mnt/lacie/scz_meta_analysis/velocity_anndatas/velocity_shin.h5ad',
 '/mnt/lacie/scz_meta_analysis/velocity_anndatas/velocity_sebastian.h5ad',
 '/mnt/lacie/scz_meta_analysis/velocity_anndatas/velocity_walsh.h5ad',
 '/mnt/lacie/scz_meta_analysis/velocity_anndatas/velocity_rao.h5ad',
 '/mnt/lacie/scz_meta_analysis/velocity_anndatas/velocity_notaras.h5ad',
 '/mnt/lacie/scz_meta_analysis/velocity_anndatas/velocity_fernando.h5ad']

In [ ]:
## Load and Filter All Remaining Manuscripts ##

# Remove Cellbendered Barcodes From Teh Other Manuscripts # 
# will do sebastian again, thats fine #

scz_adata = sc.read_h5ad("/mnt/sdb/scz_meta_analysis_processed/anndata_objs/integrated_adata_annotated_w_celltypist_scanvi.h5ad",
                        backed="r")

master_set = set(scz_adata.obs_names)

# files = glob.glob("/mnt/lacie/scz_meta_analysis/velocity_anndatas/*.h5ad")

files = ['/mnt/lacie/scz_meta_analysis/velocity_anndatas/velocity_purcell.h5ad',
         '/mnt/lacie/scz_meta_analysis/velocity_anndatas/velocity_fernando.h5ad']

files = ['/mnt/lacie/scz_meta_analysis/velocity_anndatas/velocity_fernando.h5ad']

for f in tqdm(files):

    print(f)

    ad = sc.read_h5ad(f, backed="r")

    mask = ad.obs_names.isin(master_set)
    if mask.sum() == 0:
        del ad
        continue

    #ad_filt = ad[mask].to_memory()   
    ad_filt = ad[mask]

    tmp_path = f + ".tmp.h5ad"
    
    ad_filt.copy(filename=tmp_path)
    # ad_filt.write_h5ad(tmp_path)

    del ad, ad_filt
    gc.collect()

    # atomic replace
    os.replace(tmp_path, f)

  0%|                                                             | 0/1 [00:00<?, ?it/s]

/mnt/lacie/scz_meta_analysis/velocity_anndatas/velocity_fernando.h5ad


In [ ]:
## Final Concatenation 

In [2]:
files = ['/mnt/lacie/scz_meta_analysis/velocity_anndatas/velocity_fernando.h5ad']

scz_adata = sc.read_h5ad("/mnt/sdb/scz_meta_analysis_processed/anndata_objs/integrated_adata_annotated_w_celltypist_scanvi.h5ad",
                        backed="r")

master_set = set(scz_adata.obs_names)

for f in tqdm(files):

    print(f)

    ad = sc.read_h5ad(f, backed="r")

    # Filter directly without creating a large mask in memory
    ad_filt = ad[ad.obs_names.isin(master_set), :]

    if ad_filt.shape[0] == 0:
        del ad
        continue

    print(ad.shape)
    print(ad_filt.shape)

    tmp_path = f + ".tmp.h5ad"
    
    # Directly write filtered data to disk
    ad_filt.write_h5ad(tmp_path)

    del ad, ad_filt
    gc.collect()

    # Atomic replace
    os.replace(tmp_path, f)

  0%|                                                            | 0/1 [00:00<?, ?it/s]

/mnt/lacie/scz_meta_analysis/velocity_anndatas/velocity_fernando.h5ad


  0%|                                                            | 0/1 [30:31<?, ?it/s]


NameError: name 'master_set' is not defined